In [0]:
import re
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

medical_practitioners_source_path = (
    "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/health workforce/"
    "TABLE 9.14 Number of Government Medical Practitioners by Region- 2012 to 2023.csv"
)

# Define explicit schema — the CSV has a title row (1 field) that would otherwise
# cause Spark to infer only 1 column, dropping the other 13.
schema = StructType([
    StructField("Region", StringType(), True),
    StructField("Professionals", StringType(), True),
    StructField("2012", StringType(), True),
    StructField("2013", StringType(), True),
    StructField("2014", StringType(), True),
    StructField("2015", StringType(), True),
    StructField("2016", StringType(), True),
    StructField("2017", StringType(), True),
    StructField("2018", StringType(), True),
    StructField("2019", StringType(), True),
    StructField("2020", StringType(), True),
    StructField("2021", StringType(), True),
    StructField("2022", StringType(), True),
    StructField("2023", StringType(), True),
])

# Read with explicit schema to force all 14 columns
raw_df = (
    spark.read
         .option("header", False)
         .schema(schema)
         .csv(medical_practitioners_source_path)
)

# Filter out the title row and header row
medical_practitioners_df = raw_df.filter(
    (~col("Region").startswith("TABLE")) & (col("Region") != "Region")
)

# Cast year columns to integers
year_columns = [c for c in medical_practitioners_df.columns if c not in ("Region", "Professionals")]
for c in year_columns:
    medical_practitioners_df = medical_practitioners_df.withColumn(c, col(c).cast("int"))

# Sanitize column names for Delta compatibility
sanitized_cols = [re.sub(r'[ ,;{}()\n\t=]', '_', c) for c in medical_practitioners_df.columns]
medical_practitioners_df = medical_practitioners_df.toDF(*sanitized_cols)

(
    medical_practitioners_df.write
                           .format("delta")
                           .mode("overwrite")
                           .option("overwriteSchema", "true")
                           .saveAsTable(
                               "ph_public_health.bronze.government_medical_practitioners_by_region"
                           )
)

display(medical_practitioners_df)